# Phase 18 — Evaluation harness + SchemaRAG Table 5 ablation

Runs `scripts/run_baseline.py` (CP1 floor) and `scripts/run_eval.py` (full pipeline + 4-way ablation) end-to-end on whatever GPU this session gets. Produces the EX numbers the plan has been waiting on since Phase 15 — nothing here has been run on real held-out data yet, so treat every number this notebook prints as the first look.

SQL uses `Data/cot_data/sql_dev_eval_full.json` — a real held-out Spider **dev**-split set (1034 questions, live SchemaLinker `key_fields`, built in Phase 15A). NoSQL has no native held-out split, so per the README it evaluates on `Data/cot_data/nosql_cot_train.json` — the Generator was fine-tuned on that file, so the NoSQL EX number measures memorization, not generalization. Report it as such in Phase 19.

## 0. Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        print(f"GPU {i}: {name}  ({total_gb:.1f} GB)")
        # Mirrors the 20GiB cut in src/generator/infer.py -- keep the two in sync.
        if total_gb >= 20:
            print("  -> >=20GiB (A100/L4-class). GeneratorInfer loads full bf16, no quantization.")
        else:
            print("  -> <20GiB (T4-class). GeneratorInfer loads int8 via bitsandbytes.")
else:
    raise RuntimeError("No GPU visible -- set Runtime > Change runtime type > GPU before continuing.")

## 1. Clone repo + install dependencies

In [ ]:
!git clone https://github.com/kethansplunk/Codegen.git
%cd Codegen
!pip install -q torch transformers peft sqlparse pyyaml FlagEmbedding chromadb openai python-dotenv pymongo bitsandbytes

## 2. Mount Drive and load checkpoints (both tracks)

Loads SAR + Generator checkpoints for **both** SQL and NoSQL from Drive. If your Drive layout differs from `codegen/checkpoints/{sar,generator}_{sql,nosql}`, adjust `DRIVE` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/codegen'
os.makedirs('models', exist_ok=True)

for name in ['sar_sql', 'generator_sql', 'sar_nosql', 'generator_nosql']:
    dst = f'models/{name}'
    if not os.path.exists(dst):
        os.symlink(f'{DRIVE}/checkpoints/{name}', dst)

!ls -la models/sar_sql models/generator_sql models/sar_nosql models/generator_nosql

In [ ]:
# Safety net: force sar.backend to memory. ChromaDB's PersistentClient can't open
# an index over a Google Drive FUSE mount, so this avoids that failure mode entirely.
text = open('configs/config.yaml').read()
text = text.replace('backend: chroma', 'backend: memory')
open('configs/config.yaml', 'w').write(text)
!grep -A1 "^sar:" configs/config.yaml | head -3

## 3. Spider SQLite databases

Needed for SQL EX scoring (executing gold + predicted queries), and as the source data for the MongoDB conversion below. Uploaded once as a zip to Drive.

In [ ]:
!cp /content/drive/MyDrive/codegen/checkpoints/spider_database.zip /content/Codegen/
!unzip -q /content/Codegen/spider_database.zip -d /content/Codegen/Data/Spider/
!ls /content/Codegen/Data/Spider/database | wc -l

## 4. MongoDB setup (needed for NoSQL EX)

**Important**: `Data/mongodb/*.json` schema-cache files are git-tracked from an earlier run on a different machine. `convert_all()` treats their existence as "already converted" and will silently skip real data insertion into this fresh session's empty `mongod` if we don't clear them first. Skip this section if you're only running the SQL track.

In [ ]:
# Install and start mongod
!apt-get install -y mongodb >/dev/null 2>&1 || (curl -fsSL https://pgp.mongodb.com/server-7.0.asc | sudo gpg -o /usr/share/keyrings/mongodb-server-7.0.gpg --dearmor && echo "deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list && apt-get update -qq && apt-get install -y mongodb-org)
!mkdir -p /data/db
import subprocess, time
subprocess.Popen(['mongod', '--dbpath', '/data/db', '--logpath', '/var/log/mongod.log', '--fork'])
time.sleep(3)
!tail -n 5 /var/log/mongod.log

In [ ]:
import shutil
shutil.rmtree('Data/mongodb', ignore_errors=True)   # force a real reconversion, see note above

from src.mongodb_converter import convert_all
convert_all(
    db_root="Data/Spider/database",
    fk_graph_dir="Data/fk_graphs",
    schema_cache_dir="Data/mongodb",
)

In [ ]:
# Verify real data landed (not just schema cache) before trusting any NoSQL EX result
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
print(len(client.list_database_names()), client.list_database_names()[:10])
print("formula_1.drivers count:", client['formula_1']['drivers'].count_documents({}))   # must be > 0

## 5. DeepSeek API key

Needed by the `SchemaLinker` call inside every `run_eval.py` / `run_baseline.py` invocation below. Paste your real key in place of the placeholder, then re-run this cell — it will not overwrite a key that's already there. **Never commit this cell with a real key filled in.**

In [ ]:
from pathlib import Path

PLACEHOLDER = 'your_actual_key_here'
env = Path('.env')
existing = env.read_text() if env.exists() else ''

if 'DEEPSEEK_API_KEY=' in existing and PLACEHOLDER not in existing:
    print('.env already contains a DEEPSEEK_API_KEY -- left untouched.')
else:
    env.write_text(f'DEEPSEEK_API_KEY={PLACEHOLDER}\n')
    print('Wrote .env with a placeholder. Edit it (or this cell) with your real key, then re-run.')

## 6. Pull the held-out SQL eval set from Drive

`sql_dev_eval_full.json` (1034 Spider dev questions, real SchemaLinker `key_fields`) was built locally in Phase 15A. Upload it to Drive from your Mac first if it isn't there already.

In [ ]:
!cp /content/drive/MyDrive/codegen/sql_dev_eval_full.json Data/cot_data/sql_dev_eval_full.json
!wc -l Data/cot_data/sql_dev_eval_full.json

## 7. CP1 baseline — codegen-350M floor (SQL)

No SchemaLinker, no SAR, no POSG, no fine-tuning — just the schema and question prompted into a plain pretrained code LM. Plan expects ~45–55% EX; this is the gap the full SchemaRAG pipeline is claimed to close.

In [ ]:
%%time
!python -m scripts.run_baseline --data Data/cot_data/sql_dev_eval_full.json --n 100

## 8. SQL track — smoke test before the full run

Timed on a small `--n` so you can extrapolate wall time / compute-unit cost to the full sweep before committing budget to it: `(this cell's wall time / 10) * 100 * 3` (3 generation passes per question for the 4-way ablation, per `plan_generation_passes()`).

In [ ]:
%%time
!python -m scripts.run_eval --track sql --data Data/cot_data/sql_dev_eval_full.json \
    --n 10 --ablation all --out evaluation/results/phase18_sql_smoke.json

## 9. SQL track — full Table 5 ablation sweep (the Phase 18 headline numbers)

`full` / `no_schema_linker` / `no_sar` / `no_posg` on 100 held-out Spider dev questions. Targets: **>82% EX** for `full`, with `no_schema_linker` / `no_sar` / `no_posg` each expected to fall below it -- that gap is Table 5.

In [ ]:
%%time
!python -m scripts.run_eval --track sql --data Data/cot_data/sql_dev_eval_full.json \
    --n 100 --ablation all --out evaluation/results/phase18_sql_full.json

## 9b. Why is `full` at 79.0% instead of >82%? — failure breakdown

Pure local analysis of the report written in section 9 — no GPU/model calls, so this is free to re-run. Buckets every scored question by the same `_is_hard()` heuristic `--hard` uses (multi-JOIN / subquery / GROUP BY) and prints the actual failing question/gold/predicted query for anything `full` got wrong, so you can eyeball whether the shortfall concentrates in complex queries or is spread evenly.

In [ ]:
import json
from scripts.run_eval import _is_hard

report  = json.load(open("evaluation/results/phase18_sql_full.json"))
results = report["results"]

def bucket(r):
    return "hard" if _is_hard({"sql": r["gold"]}, "sql") else "easy"

buckets = {"hard": {"n": 0, "wrong": 0}, "easy": {"n": 0, "wrong": 0}}
failures = []
for r in results:
    full = r["ablations"]["full"]
    if not full["ex_eligible"]:
        continue
    b = bucket(r)
    buckets[b]["n"] += 1
    if full["ex"] == 0.0:
        buckets[b]["wrong"] += 1
        failures.append(r)

print(f"{'bucket':6} {'n':>4} {'EX':>8}")
for b, d in buckets.items():
    n, wrong = d["n"], d["wrong"]
    ex = (n - wrong) / n if n else None
    print(f"{b:6} {n:4} {f'{ex:.1%}' if ex is not None else 'n/a':>8}")

print(f"\n{len(failures)} misses out of {sum(d['n'] for d in buckets.values())} scored questions\n")
for r in failures:
    full = r["ablations"]["full"]
    print(f"[{bucket(r)}] {r['question']}  (db={r['db_name']})")
    print(f"    gold: {r['gold']}")
    print(f"    pred: {full['query']}")
    if full["pred_error"]:
        print(f"    error: {full['pred_error']}")
    print()

## 10. NoSQL track — smoke test

Requires the live `mongod` + loaded databases from section 4. Uses the **train** split (no held-out NoSQL set exists) — see the caveat at the top of this notebook.

In [ ]:
%%time
!python -m scripts.run_eval --track nosql --data Data/cot_data/nosql_cot_train.json \
    --n 10 --ablation all --out evaluation/results/phase18_nosql_smoke.json

## 11. NoSQL track — full ablation sweep

Target: **>60% EX** for `full`. Remember this number is on the training split, so report it in Phase 19 as an upper bound / memorization check, not a generalization result.

In [ ]:
%%time
!python -m scripts.run_eval --track nosql --data Data/cot_data/nosql_cot_train.json \
    --n 100 --ablation all --out evaluation/results/phase18_nosql_full.json

## 12. Copy reports back to Drive

`evaluation/results/*.json` lives only in this ephemeral Colab session -- copy it out before disconnecting, or Phase 19 error analysis has nothing to read.

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/codegen/evaluation_results', exist_ok=True)
!cp evaluation/results/*.json /content/drive/MyDrive/codegen/evaluation_results/
!ls -la /content/drive/MyDrive/codegen/evaluation_results/

## 13. Free the GPU when you're done

Units keep burning as long as the runtime stays connected, even idle. Run this once you've confirmed the reports landed on Drive above.

In [ ]:
from google.colab import runtime
runtime.unassign()